In [ ]:
# LSTM + Attention Model

import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import evaluate

os.environ["WANDB_DISABLED"] = "true"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# 1. LOAD DATASET
dataset = load_dataset(" ") # Fill in

train_data = dataset["train"]
test_data = dataset["test"]


In [ ]:
# 2. TOKENIZATION

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

tokenized_train = train_data.map(tokenize_batch, batched=True)
tokenized_test = test_data.map(tokenize_batch, batched=True)

tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


In [ ]:
# 3. PYTORCH DATASETS

train_loader = DataLoader(tokenized_train, batch_size=8, shuffle=True)
test_loader = DataLoader(tokenized_test, batch_size=8)

In [ ]:
# 4. DEFINE LSTM + ATTENTION MODEL

class Attention(nn.Module):
    """
    Additive (Bahdanau) attention
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)  # for biLSTM
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_output):
        # lstm_output: [batch, seq, hidden*2]
        score = torch.tanh(self.attn(lstm_output))  # [batch, seq, hidden]
        attention_weights = torch.softmax(self.v(score), dim=1)  # [batch, seq, 1]
        context = (attention_weights * lstm_output).sum(dim=1)   # [batch, hidden*2]
        return context, attention_weights


class LSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)  # [batch, seq, embed]
        lstm_output, _ = self.lstm(embedded)  # [batch, seq, hidden*2]

        # Apply attention
        context, attn_weights = self.attention(lstm_output)

        logits = self.fc(context)  # [batch, num_classes]
        return logits

# Model initialization
vocab_size = tokenizer.vocab_size

model = LSTMAttentionClassifier(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=128,
    num_classes=2
).to(device)

In [ ]:
# 5. TRAINING SETUP

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
num_epochs = 3

accuracy_metric = evaluate.load("accuracy")

In [ ]:
# 6. TRAINING LOOP

print("Training LSTM+Attention...")
train_start = time.time()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss = {total_loss/len(train_loader):.4f}")

if torch.cuda.is_available():
    print(f"GPU Memory After Training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

train_time = time.time() - train_start
print(f"Training time: {train_time:.2f} seconds")



In [ ]:
# 7. EVALUATION LOOP

print("\nEvaluating...")

eval_start = time.time()
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids)
        preds = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

accuracy = accuracy_metric.compute(predictions=all_preds, references=all_labels)

if torch.cuda.is_available():
    print(f"GPU Memory After Evaluation: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

eval_time = time.time() - eval_start

print(f"Inference time: {eval_time:.2f} seconds")
print("Accuracy:", accuracy)